In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
)
from context_flux_no.models.multiphysics.hyperfluxfno import (
    HyperNeuralOperator,
)
from jaxtyping import Array, Float, PRNGKeyArray


jax.config.update("jax_default_device", jax.devices("gpu")[4])

In [ ]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=2,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FluxNO",
    target_network_kwargs=dict(
        stencil_widths=(10, 10), lift_dim=128, hidden_dim=128, depth=4
    ),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)

In [ ]:
hyperfluxno = HyperNeuralOperator(
    num_spatial_dims=1,
    in_channels=2,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    target_network_type="FluxNO",
    target_network_kwargs=dict(
        stencil_widths=(10, 10), lift_dim=128, hidden_dim=128, depth=4
    ),
    lift_dim=64,
    width_hyper=128,
    blocks_hyper=16,
    key=jax.random.key(0),
)
print(hyperfluxno.num_parameters() / 1e6)

In [ ]:
@eqx.filter_jit
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


test_data = jax.random.normal(jax.random.key(0), (512, 21, 2, 100))

In [ ]:
hyperfluxno(test_data[0, :20], (0.1, 0.01))